# 생성 지식 프롬프팅 (Generate Knowledge Prompting)

생성 지식 프롬프팅은 언어 모델에게 외부 지식을 생성하도록 한 다음, 그 지식을 활용하여 질문에 답하게 하는 기법입니다. 이 방식을 통해 모델이 더 정확하고 사실에 기반한 응답을 할 수 있도록 돕습니다.

해당 예제를 기반으로도 충분한 학습을 할 수 있지만 상용 ChatGPT 혹은 Claude를 통해 예제를 수행해보세요.
Generate Knowledge Prompting은 stateful conversation에 더 유용한 기술입니다.

In [1]:
# 필요한 라이브러리 및 모듈 임포트
import os
import sys
from dotenv import load_dotenv

# utils 디렉토리의 helpers.py 모듈을 임포트하기 위한 경로 설정
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils.helpers import get_completion

# .env 파일 로드
load_dotenv()

True

## 1. 생성 지식 프롬프팅 함수 정의

먼저 질문에 관련된 지식을 생성하고, 그 지식을 바탕으로 답변하는 함수를 정의합니다.

In [2]:
def generate_knowledge(question):
    """
    질문에 관련된 지식을 생성하는 함수
    """
    knowledge_prompt = f"""
    다음 질문에 관한 배경 지식을 생성해주세요. 이 지식은 질문에 답변하는 데 도움이 될 수 있는 사실적인 정보여야 합니다:
    
    질문: {question}
    
    지식:
    """
    
    return get_completion(knowledge_prompt)

def answer_with_knowledge(question, knowledge):
    """
    생성된 지식을 활용하여 질문에 답변하는 함수
    """
    answer_prompt = f"""
    질문: {question}
    
    지식: {knowledge}
    
    위 지식을 활용하여 질문에 답변해주세요. 답변은 "예" 또는 "아니오"로 시작한 후, 설명을 추가해주세요:
    """
    
    return get_completion(answer_prompt)

## 2. 원문 예제: 골프 관련 질문

원문에서 사용된 골프 관련 질문으로 먼저 실험해봅니다. 지식 없이 직접 답변한 결과와 비교해보겠습니다.

In [3]:
# 원문 예제 질문
question = "골프에서는 점수를 높게 받는 것이 목표인가요?"
print(f"질문: {question}\n")

# 지식 없이 직접 답변
direct_prompt = f"질문: {question}\n\n답변은 '예' 또는 '아니오'로 시작한 후, 설명을 추가해주세요:"
print("직접 답변 (지식 없이):")
direct_answer = get_completion(direct_prompt)
print(direct_answer)

질문: 골프에서는 점수를 높게 받는 것이 목표인가요?

직접 답변 (지식 없이):
아니오. 골프에서는 점수를 낮게 받는 것이 목표입니다. 골프는 코스를 완료하는 데 필요한 타수, 즉 스트로크 수가 적을수록 더 좋은 점수를 받게 되는 스포츠입니다. 따라서 각 홀에서 가능한 적은 타수로 공을 홀에 넣는 것이 목표입니다. 최종적으로 모든 홀의 타수를 합산하여 가장 낮은 점수를 기록한 선수가 승리하게 됩니다.


이제 생성 지식을 활용한 답변을 얻어보겠습니다.

In [4]:
# 지식 생성
print("생성된 지식:")
knowledge = generate_knowledge(question)
print(knowledge)

# 생성된 지식으로 답변
print("\n지식 기반 답변:")
answer = answer_with_knowledge(question, knowledge)
print(answer)

생성된 지식:
골프는 상대적으로 독특한 점수 체계를 가진 스포츠입니다. 골프의 기본 목표는 코스에 설정된 홀을 가능한 한 적은 타수로 공을 넣는 것입니다. 따라서, 골프에서 점수는 낮을수록 좋습니다. 각 홀마다 정해진 기준 타수인 "파(par)"가 있으며, 플레이어는 이 파보다 적은 타수로 홀을 완료하는 것을 목표로 합니다. 

골프 경기에는 몇 가지 주요 점수 용어가 있습니다:

1. **버디(Birdie)**: 파보다 1타 적은 타수로 홀을 완료한 경우.
2. **이글(Eagle)**: 파보다 2타 적은 타수로 홀을 완료한 경우.
3. **알바트로스(Albatross) 또는 더블 이글(Double Eagle)**: 파보다 3타 적은 타수로 홀을 완료한 경우.
4. **보기(Bogey)**: 파보다 1타 많은 타수로 홀을 완료한 경우.
5. **더블 보기(Double Bogey)**: 파보다 2타 많은 타수로 홀을 완료한 경우.

골프 경기의 승자는 일반적으로 전체 코스를 완료하는 데 가장 적은 총 타수를 기록한 플레이어입니다. 이처럼 골프에서는 점수를 낮게 유지하는 것이 목표입니다.

지식 기반 답변:
아니오. 골프에서는 점수를 높게 받는 것이 목표가 아닙니다. 골프의 목표는 각 홀을 가능한 한 적은 타수로 완료하여 전체 코스의 총 점수를 낮게 유지하는 것입니다. 따라서, 골프에서 점수는 낮을수록 좋으며, 파보다 적은 타수로 홀을 완료하는 것이 이상적입니다. 예를 들어, 파보다 1타 적은 버디, 2타 적은 이글 등의 점수를 목표로 합니다. 최종적으로, 가장 적은 총 타수를 기록한 플레이어가 승자가 됩니다.


## 3. 여러 지식 생성 및 비교

같은 질문에 대해 여러 지식을 생성하고, 각 지식이 답변에 어떤 영향을 미치는지 비교해보겠습니다.

In [5]:
print("여러 지식 생성 실험:\n")

# 첫 번째 지식 생성
print("지식 1 생성:")
knowledge1 = generate_knowledge(question)
print(knowledge1)

# 두 번째 지식 생성
print("\n지식 2 생성:")
knowledge2 = generate_knowledge(question)
print(knowledge2)

# 첫 번째 지식 기반 답변
print("\n지식 1 기반 답변:")
answer1 = answer_with_knowledge(question, knowledge1)
print(answer1)

# 두 번째 지식 기반 답변
print("\n지식 2 기반 답변:")
answer2 = answer_with_knowledge(question, knowledge2)
print(answer2)

여러 지식 생성 실험:

지식 1 생성:

골프는 독특한 채점 방식을 가진 스포츠입니다. 일반적인 스포츠와 달리, 골프에서는 점수를 낮게 받는 것이 목표입니다. 골프의 기본 목표는 가능한 한 적은 타수로 공을 홀에 넣는 것입니다. 각 홀마다 '파'라는 기준 타수가 설정되어 있으며, 플레이어는 이 파보다 적은 타수로 홀을 완료하는 것을 목표로 합니다. 예를 들어, 파 4의 홀에서 3타 만에 공을 넣으면 '버디'를 기록하며, 이는 좋은 성적으로 간주됩니다. 반면, 파보다 많은 타수를 기록하면 '보기' 또는 '더블 보기' 등으로 불리며, 이는 상대적으로 좋지 않은 성적입니다. 전체 라운드의 최종 점수는 모든 홀에서 기록한 타수를 합산하여 계산되며, 이 점수가 낮을수록 더 좋은 성과를 의미합니다. 이처럼 골프에서는 점수를 낮추는 것이 승리의 핵심입니다.

지식 2 생성:
골프는 스코어가 낮을수록 좋은 스포츠입니다. 골프에서는 각 홀에서 공을 최대한 적은 타수로 홀에 넣는 것이 목표입니다. 따라서, 최종적으로 18홀을 도는 동안의 총 타수를 합산했을 때 그 수가 적을수록 좋은 점수를 얻게 됩니다. 일반적으로 "파(par)"라는 기준 타수가 있으며, 각 홀마다 파가 정해져 있습니다. 플레이어의 목표는 파 이하의 타수로 홀을 마치는 것입니다. 예를 들어, 파 4인 홀에서 3타로 홀 아웃을 하면 "버디(birdie)"라고 부르며 이는 좋은 성과로 여겨집니다. 반대로, 파보다 많은 타수로 홀을 마치면 "보기(bogey)"라고 하며, 이는 덜 좋은 결과로 간주됩니다. 따라서 골프에서는 점수가 낮을수록 목표에 더 가까워지는 것입니다.

지식 1 기반 답변:
아니오, 골프에서는 점수를 높게 받는 것이 목표가 아닙니다. 골프의 목표는 가능한 한 적은 타수로 공을 홀에 넣는 것입니다. 각 홀에는 '파'라는 기준 타수가 있으며, 이보다 적은 타수로 홀을 완료하는 것이 좋은 성적을 의미합니다. 따라서, 골프에서는 점수를 낮추는 것이 승리의 핵심입니다.

지식 2 기반 답변:
아니오. 골프

## 4. 추가 질문 실험

다른 질문들에 대해서도 생성 지식 프롬프팅을 적용해보겠습니다.

In [ ]:
# 추가 질문 목록
additional_questions = [
    "독서는 뇌 발달에 도움이 되나요?",
    "지구는 태양계에서 가장 큰 행성인가요?"
]

# 각 질문에 대해 지식 생성 및 답변 비교
for question in additional_questions:
    print(f"질문: {question}\n")
    
    # 지식 없이 직접 답변
    direct_prompt = f"질문: {question}\n\n답변은 '예' 또는 '아니오'로 시작한 후, 설명을 추가해주세요:"
    print("직접 답변 (지식 없이):")
    direct_answer = get_completion(direct_prompt)
    print(direct_answer)
    
    # 지식 생성
    print("\n생성된 지식:")
    knowledge = generate_knowledge(question)
    print(knowledge)
    
    # 생성된 지식으로 답변
    print("\n지식 기반 답변:")
    answer = answer_with_knowledge(question, knowledge)
    print(answer)

질문: 독서는 뇌 발달에 도움이 되나요?

직접 답변 (지식 없이):
예. 독서는 뇌 발달에 도움이 됩니다. 독서를 통해 다양한 정보를 처리하고 이해하는 과정에서 뇌의 여러 영역이 활성화됩니다. 이는 언어 능력, 집중력, 기억력 및 문제 해결 능력을 향상시키는 데 기여합니다. 특히, 어린 시절의 독서는 언어 발달과 인지 능력 향상에 중요한 역할을 하며, 성인의 경우에도 독서는 지속적인 뇌 건강 유지와 인지 기능 강화에 긍정적인 영향을 미칩니다.

생성된 지식:
독서는 뇌 발달에 긍정적인 영향을 미치는 여러 가지 이유가 있습니다.

1. **인지 능력 향상**: 독서는 어휘력과 이해력을 증진시키며, 복잡한 문장을 이해하고 스토리를 따라가는 과정에서 인지 능력이 발달합니다.

2. **집중력과 주의력 강화**: 독서는 독자가 텍스트에 집중하고 긴 시간 동안 주의를 기울이게 합니다. 이는 집중력과 주의력을 강화하는 데 기여합니다.

3. **창의력과 상상력 증진**: 독서를 통해 다양한 상황과 캐릭터를 접하면서 독자의 상상력과 창의력이 자극됩니다. 이는 문제 해결 능력을 향상시키는 데에도 도움이 됩니다.

4. **뇌의 연결성 강화**: 연구에 따르면 독서는 뇌의 여러 부분 간의 연결성을 강화시킵니다. 특히, 이야기 구조를 파악하고 이해하는 과정에서 좌뇌와 우뇌의 협력이 이루어집니다.

5. **정서적 공감 능력 발달**: 독서는 다양한 인물과 상황을 경험하게 하여 독자가 타인의 감정을 이해하고 공감하는 능력을 기르는 데 도움을 줍니다.

6. **스트레스 감소**: 독서는 스트레스를 줄이고 마음을 안정시키는 데 효과적입니다. 이는 전반적인 정신 건강을 개선하는 데 기여합니다.

이러한 이유들로 인해 독서는 어린이뿐만 아니라 성인에게도 뇌 발달과 정신 건강에 긍정적인 영향을 미친다고 할 수 있습니다.

지식 기반 답변:
예, 독서는 뇌 발달에 도움이 됩니다. 독서는 어휘력과 이해력을 높이고, 인지 능력을 향상시키며, 집중력과 주의력을 강화합니다. 또한, 창의력과 상

## 5. 논쟁의 여지가 있는 질문 실험

논쟁의 여지가 있는 질문에 대해 생성 지식 프롬프팅의 효과를 확인해보겠습니다.

In [ ]:
# 논쟁의 여지가 있는 질문으로 실험
challenging_question = "인공지능은 인간의 일자리를 모두 대체할까요?"
print(f"질문: {challenging_question}\n")

# 지식 없이 직접 답변
direct_prompt = f"질문: {challenging_question}\n\n답변은 '예' 또는 '아니오'로 시작한 후, 설명을 추가해주세요:"
print("직접 답변 (지식 없이):")
direct_answer = get_completion(direct_prompt)
print(direct_answer)

# 지식 생성
print("\n생성된 지식:")
knowledge = generate_knowledge(challenging_question)
print(knowledge)

# 생성된 지식으로 답변
print("\n지식 기반 답변:")
answer = answer_with_knowledge(challenging_question, knowledge)
print(answer)

## 6. 답변 비교 분석

지식 없이 직접 답변한 결과와 지식 기반 답변의 차이점을 분석해보겠습니다.

In [ ]:
# 답변 비교 분석
print("두 답변의 차이점 분석:")
comparison_prompt = f"""
다음 두 답변의 주요 차이점을 분석해주세요:

직접 답변: {direct_answer}

지식 기반 답변: {answer}

분석:
"""
comparison = get_completion(comparison_prompt)
print(comparison)

## 7. 결론

생성 지식 프롬프팅은 모델이 질문에 답하기 전에 관련 지식을 먼저 생성하도록 함으로써 더 정확하고 사실에 기반한 응답을 제공할 수 있게 합니다. 이 기법은 특히 다음과 같은 경우에 유용합니다:

1. 사실 확인이 필요한 질문
2. 상식적 지식을 요구하는 질문
3. 복잡한 주제에 대한 질문
4. 논쟁의 여지가 있는 질문

실험 결과를 통해 생성 지식 프롬프팅이 모델의 환각(hallucination)을 줄이고, 더 신뢰할 수 있는 답변을 생성하는 데 기여함을 확인할 수 있었습니다. 다만, 생성된 지식 자체의 정확성에 따라 최종 답변의 품질이 좌우될 수 있다는 점은 유의해야 합니다.